# 01. 데이터 품질 점검과 전처리

**입력**: `dataSet/student_performance_dataset.csv` (Kaggle, 3,000행 × 32열)
**출력**: `dataSet/student_clean.csv` (2,737행 × 30열) — 뒤의 모든 노트북이 이 파일을 쓴다. 점검 결과 표는 `out/01_*.csv`.

데이터 출처: Kaggle 「Student Academic Performance Prediction」 (aiexplorer77, MIT).
실제 학생을 모사한 **합성 데이터**라는 점을 분석 내내 기억해야 한다.

이 노트북에서 하는 일
1. 자료형·중복·결측·이상치 점검
2. 불필요 컬럼 2개 제거 (`student_id`, `performance_category`)
3. 결측치 177행 제거 — 제거 전후 분포가 달라지지 않았는지 검정으로 확인
4. 하루 시간 변수 6개의 합이 24시간을 넘는 86행 제거 (논리적 이상치)

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.width', 160)

df = pd.read_csv('dataSet/student_performance_dataset.csv')
print(df.shape)
df.head()

(3000, 32)


,student_id,age,gender,city_type,study_hours_per_day,deep_work_sessions,assignment_completion_rate,attendance_percentage,social_media_hours,doomscrolling_before_sleep,...,family_support,financial_stress,learning_style,career_goal,productivity_after_midnight,revision_efficiency,burnout_risk,consistency_score,final_exam_score,performance_category
0,0,21,Female,Semi-Urban,3.2,7,100,70,3.8,0,...,10,6,Audio,Engineering,9,1,10,3,32,Low
1,1,19,Female,Semi-Urban,3.9,2,46,70,2.6,1,...,2,6,Practical,Business,6,10,4,6,59,Medium
2,2,16,Female,Urban,4.3,7,54,57,4.3,1,...,1,9,Visual,Engineering,7,9,8,1,34,Low
3,3,19,Male,Semi-Urban,5.3,1,78,90,1.7,0,...,5,10,Reading,Medical,7,7,9,10,60,Medium
4,4,17,Female,Urban,4.1,3,100,81,2.6,0,...,5,6,Visual,Medical,4,10,7,9,77,High


## 1. 자료형과 중복

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   student_id                   3000 non-null   int64  
 1   age                          3000 non-null   int64  
 2   gender                       3000 non-null   str    
 3   city_type                    3000 non-null   str    
 4   study_hours_per_day          3000 non-null   float64
 5   deep_work_sessions           3000 non-null   int64  
 6   assignment_completion_rate   3000 non-null   int64  
 7   attendance_percentage        3000 non-null   int64  
 8   social_media_hours           2940 non-null   float64
 9   doomscrolling_before_sleep   3000 non-null   int64  
 10  notification_distractions    3000 non-null   int64  
 11  ai_tool_usage_hours          3000 non-null   float64
 12  gaming_hours                 3000 non-null   float64
 13  stress_level                 

In [3]:
print('중복 행:', df.duplicated().sum())
print('student_id 고유값:', df['student_id'].nunique(), '/', len(df))

중복 행: 0
student_id 고유값: 3000 / 3000


## 2. 불필요 컬럼 제거

- `student_id`: 단순 번호라 정보가 없다.
- `performance_category`: 종속변수(`final_exam_score`)를 구간으로 나눈 파생 변수.
  정답을 그대로 담고 있어 설명변수로 쓰면 **데이터 누수**가 된다.

In [4]:
# performance_category가 정말 점수에서 나온 것인지 확인
df.groupby('performance_category')['final_exam_score'].agg(['min', 'max', 'count'])

,min,max,count
performance_category,,,
High,75,100,444
Low,0,49,1196
Medium,50,74,1360


In [5]:
df = df.drop(columns=['student_id', 'performance_category'])

# 문자열 컬럼은 category 타입으로 (이후 더미 변환이 편하다)
str_cols = df.select_dtypes(exclude='number').columns.tolist()
df[str_cols] = df[str_cols].astype('category')
print('범주형:', str_cols)

범주형: ['gender', 'city_type', 'mental_state', 'learning_style', 'career_goal']


## 3. 결측치

결측은 세 변수에 60건씩, 모두 177행(5.9%)이다.
그냥 지워도 되는지 확인하려면 **결측이 있는 행과 없는 행의 다른 변수 분포가 같은지** 보면 된다.
같다면 결측은 무작위에 가깝고, 제거해도 표본이 한쪽으로 치우치지 않는다.

In [6]:
na_count = df.isna().sum()
print(na_count[na_count > 0])
print('결측 포함 행:', df.isna().any(axis=1).sum(), f'({df.isna().any(axis=1).mean():.1%})')

social_media_hours    60
focus_score           60
sleep_hours           60
dtype: int64
결측 포함 행: 177 (5.9%)


In [7]:
# 결측 행 vs 비결측 행 — 변수별로 분포 차이 검정
# 연속형은 Mann-Whitney U, 범주형은 카이제곱 독립성 검정
has_na = df.isna().any(axis=1)
rows = []
for col in df.columns:
    a = df.loc[has_na, col].dropna()
    b = df.loc[~has_na, col].dropna()
    if col in str_cols:
        table = pd.crosstab(has_na, df[col])
        stat, p, _, _ = stats.chi2_contingency(table)
        method = 'chi2'
    else:
        stat, p = stats.mannwhitneyu(a, b)
        method = 'Mann-Whitney U'
    rows.append({'변수': col, '검정': method, 'p-value': round(p, 3)})

na_test = pd.DataFrame(rows)
print('p < 0.05 인 변수 수:', (na_test['p-value'] < 0.05).sum(), '/', len(na_test))
na_test

p < 0.05 인 변수 수: 3 / 30


,변수,검정,p-value
0,age,Mann-Whitney U,0.790
1,gender,chi2,0.604
2,city_type,chi2,0.476
3,study_hours_per_day,Mann-Whitney U,0.010
4,deep_work_sessions,Mann-Whitney U,0.957
5,assignment_completion_rate,Mann-Whitney U,0.182
6,attendance_percentage,Mann-Whitney U,0.426
7,social_media_hours,Mann-Whitney U,0.081
8,doomscrolling_before_sleep,Mann-Whitney U,0.112
9,notification_distractions,Mann-Whitney U,0.564


30개 변수 중 p < 0.05 인 것은 `internet_quality`(0.006), `study_hours_per_day`(0.010), `ai_tool_usage_hours`(0.035) 세 개다.
검정을 30번 하면 우연히 p < 0.05가 나오는 변수가 1~2개쯤 생기는 게 정상이고,
다중검정 보정(Bonferroni, 0.05/30 = 0.0017)을 적용하면 셋 다 유의하지 않다.
결측은 무작위에 가깝다고 보고 행을 제거한다.

In [8]:
na_test.to_csv('out/01_결측_제거전후_검정.csv', index=False)

# 결측 행 제거
df = df.dropna().reset_index(drop=True)
print(df.shape)

(2823, 30)


## 4. 논리적 이상치 — 하루가 24시간을 넘는 행

"하루 평균 ~시간"으로 조사된 변수가 6개 있다. 이 합이 24시간을 넘으면 응답 자체가 말이 안 된다.
같은 활동이 두 변수에 중복 기록됐을 수도 있지만 측정 방식을 알 수 없으니 보수적으로 전부 제거한다.

통계적 이상치(IQR 기준)는 논리적으로 틀린 값이 아니므로 여기서는 손대지 않는다.
경계값으로 대체했을 때 회귀 결과가 달라지는지는 04 노트북에서 비교한다.

In [9]:
time_cols = ['study_hours_per_day', 'social_media_hours', 'ai_tool_usage_hours',
             'gaming_hours', 'sleep_hours', 'physical_activity_hours']

total_hours = df[time_cols].sum(axis=1)
over_24 = total_hours > 24
print('24시간 초과 행:', over_24.sum(), f'({over_24.mean():.1%})')
print(total_hours.describe().round(2))

24시간 초과 행: 86 (3.0%)
count    2823.00
mean       18.11
std         3.04
min         8.20
25%        16.00
50%        18.00
75%        20.10
max        29.50
dtype: float64


In [10]:
df = df[~over_24].reset_index(drop=True)
print('최종:', df.shape)

최종: (2737, 30)


## 5. 저장

In [11]:
df.to_csv('dataSet/student_clean.csv', index=False)

# 변수 구분표 (뒤의 노트북에서 반복해서 쓴다)
summary = pd.DataFrame({
    '자료형': df.dtypes.astype(str),
    '고유값 수': df.nunique(),
    '예시': [', '.join(map(str, df[c].unique()[:4])) for c in df.columns],
})
summary.to_csv('out/01_변수목록.csv')
summary

,자료형,고유값 수,예시
age,int64,9,"21, 19, 16, 17"
gender,category,2,"Female, Male"
city_type,category,3,"Semi-Urban, Urban, Rural"
study_hours_per_day,float64,80,"3.2, 3.9, 4.3, 5.3"
deep_work_sessions,int64,8,"7, 2, 1, 3"
assignment_completion_rate,int64,61,"100, 46, 54, 78"
attendance_percentage,int64,51,"70, 57, 90, 81"
social_media_hours,float64,74,"3.8, 2.6, 4.3, 1.7"
doomscrolling_before_sleep,int64,2,"0, 1"
notification_distractions,int64,115,"107, 93, 82, 27"
